# Chapter 4 — Pipelines are values

Step 3b builds the **blessed combinator library** — plumbum-style pipeline
syntax over kernels (semantics after
[`pdum_plumbum`](https://github.com/habemus-papadum/pdum_plumbum); design
record: `docs/design/040_combinators-notes.md`). Two things set this chapter apart:

- It is a **satellite**: `src/pdum/dsl/combinators.py` (~140 counted
  lines) attaches with **zero kernel edits** — the kernel sits untouched at
  378/1150. That is the extension-locality claim, demonstrated ahead of its
  CI gate. It ships composition *machinery* plus the single concept it
  owns (`materializer`, the host boundary). Every role vocabulary — even
  `device`, the base language's neutral composable — arrives with its owning
  package, registered live below as stand-ins.
- It is the *definition layer only*. Pipelines are identities today;
  execution arrives with the compiler chain (fusion at lowering, DPS +
  launch at the backends — see combinators-notes §3b). Every stub is loud.

| Piece | What |
|---|---|
| `@op` | plumbum's `@pb` for kernels: curried stage constructors |
| `\|` | composes stages into an **inert** pipeline value |
| `>` | threads a value through, once (`value > pipeline`) |
| `stage[config]` | the bracket contract: an opaque config payload (040_combinators-notes §3c) |
| Roles + composition rules | who may compose, and what composing *means* |

Glossary terms settled: *Role, composition rule, materializer,
Stage/Pipeline.*

In [1]:
from pdum.dsl.combinators import collect, op, register_composition, register_role
from pdum.dsl.kernel.api import jit

# The satellite ships MACHINERY plus its one own concept (the materializer).
# Even "device" — the base language's neutral composable, @jit's default —
# arrives from its owner: the stdlib/core-dialect package, once lowering
# exists (ch07). Until then, the chapter registers the stand-in:
register_role("device")
register_composition("pipe", "device", "device", "fuse")  # function composition fuses


@op
def add(k):
    @jit()
    def go(x):
        return x + k

    return go


@op
def mul(k):
    @jit()
    def go(x):
        return x * k

    return go


pipeline = add(1) | mul(2) | collect
print("the pipeline: ", pipeline)
print("its identity: ", pipeline.fntype)
print("nothing ran — a pipeline is a VALUE; application is a separate act.")

the pipeline:  add.<locals>.go | mul.<locals>.go | collect
its identity:  Fn<pipe(add.<locals>.go)>(Fn<add.<locals>.go>(i64), Fn<mul.<locals>.go>(i64))
nothing ran — a pipeline is a VALUE; application is a separate act.


In [2]:
from pdum.dsl import viz

viz.install()
pipeline  # stages as chips; hover a chip for its structural identity

add.<locals>.go | mul.<locals>.go | collect

In [3]:
print("rebuilt each frame, fresh values:  ", (add(9) | mul(7) | collect).fp == pipeline.fp)
print("a stage param changes TYPE:        ", (add(1.0) | mul(2) | collect).fp == pipeline.fp)
print(
    "configured stage add(1)[64]:       ",
    (add(1)[64] | mul(2) | collect).fp == pipeline.fp,
    " <- no schemas yet: every config component value-specializes (040 §3c)",
)

rebuilt each frame, fresh values:   True
a stage param changes TYPE:         False
configured stage add(1)[64]:        False  <- no schemas yet: every config component value-specializes (040 §3c)


In [4]:
# The blessed operators FLATTEN, so grouping never even reaches the cache.
# (Compositions built outside this library still get unified by the
# content-addressed artifact tier, once lowering exists.)
a, b, c = add(1), mul(2), add(3)
print("(a|b)|c == a|(b|c):", ((a | b) | c).fp == (a | (b | c)).fp)
print("flattened parts:   ", a | b | c)

(a|b)|c == a|(b|c): True
flattened parts:    add.<locals>.go | mul.<locals>.go | add.<locals>.go


In [5]:
from pdum.dsl.combinators import IncompatibleRoles, Stage, register_role


def frag():
    @jit(kind="fragment")
    def shader():
        return (0.0, 0.0)

    return shader


# No graphics vocabulary is built in — the library owns only "device" and
# "materializer". Role vocabularies ship WITH their domains:
try:
    add(1) | frag()
except IncompatibleRoles as e:
    print("before any graphics package exists ->", e)

# Stand-in for what the WGSL backend package (ch10) will do at import time:
register_role(
    "fragment",
    hint="an entry point cannot be fused mid-pipeline; orchestration arrives with the pass runtime",
)

attempts = {
    "device | fragment": lambda: add(1) | frag(),
    "fragment | device": lambda: Stage(frag()) | add(1),
    "materializer mid-pipe": lambda: add(1) | collect | mul(2),
}
print()
for what, build in attempts.items():
    try:
        build()
    except IncompatibleRoles as e:
        print(f"{what:22s} -> {e}")

before any graphics package exists -> no Role registered for kind 'fragment'; register_role('fragment') first

device | fragment      -> no composition rule for (pipe, device, fragment) — the base language's neutral composable kernel; an entry point cannot be fused mid-pipeline; orchestration arrives with the pass runtime
fragment | device      -> no composition rule for (pipe, fragment, device) — an entry point cannot be fused mid-pipeline; orchestration arrives with the pass runtime; the base language's neutral composable kernel
materializer mid-pipe  -> a materializer ends a pipeline; nothing may follow it


Refusals are **rule lookups, not hardcoded checks**: a composition rule maps
`(op, role, role)` to the *semantics* of composing — `fuse` (inline into one
kernel), `terminal` (materialize at the boundary), later `orchestrate`
(render graphs, buffer chains). "Compatible" returns *how*, not just
*whether* — the fusion-vs-orchestration table is in
`docs/design/040_combinators-notes.md` §2.3.

And note what this chapter has now demonstrated twice: **role vocabularies
ship with their domains**. The satellite pre-enumerates nothing; `device` +
the fuse rule were registered as stand-ins for the base-language (stdlib)
package, and `fragment` for the WGSL backend package (ch10) — the same
discipline as `FragCoord` being a backend-package intrinsic, never a core
one. Terminality, by contrast, is *structural* (a `Role.terminal` flag, no
pair rules): nothing may follow a terminal; a terminal may end anything.

In [6]:
import itertools

from pdum.dsl.combinators import NotYetExecutable, set_dispatcher
from pdum.dsl.kernel.cache import FastRecord, SpecializationCache, no_compile
from pdum.dsl.kernel.valuekind import fingerprint

# From step 8 on, importing pdum.dsl installs a REAL dispatcher (batteries) —
# uninstall it for this cell so the definition/application split stays visible:
_batteries = set_dispatcher(None)
try:
    6 > (add(1) | mul(2))
except NotYetExecutable as e:
    print("without a backend:", e)

cache, serial = SpecializationCache(), itertools.count(1)


def dummy_dispatcher(pipeline, value):
    key = cache.key_for(pipeline, (fingerprint(value),), ("dummy",))
    rec = cache.get_or_compile(key, lambda: FastRecord(artifact=f"<fused kernel #{next(serial)}>"))
    return ("DeviceValue", rec.artifact)


prev = set_dispatcher(dummy_dispatcher)
print()
print("6 > pipeline ->", 6 > (add(1) | mul(2)))
with no_compile():
    for k in range(300):  # rebuild the WHOLE pipeline with fresh stage values, every step
        out = k > (add(k) | mul(k + 1))
print(f"300 rebuilt applications: compiles={cache.compiles}  hits={cache.hits}")
set_dispatcher(prev if prev is not None else _batteries)

without a backend: nothing can execute a pipeline yet — fusion lands with lowering, launch with the backends; install set_dispatcher(fn) to experiment (as ch04 does with dummy artifacts)

6 > pipeline -> ('DeviceValue', '<fused kernel #1>')
300 rebuilt applications: compiles=1  hits=300


<function __main__.dummy_dispatcher(pipeline, value)>

In [7]:
from pdum.dsl.kernel.valuekind import typeof

p = add(1) | mul(2)
print(typeof(p))
print(typeof((p, add(3) | mul(4))), " <- pipelines compose as values, like any Handle")

Fn<pipe(add.<locals>.go)>(Fn<add.<locals>.go>(i64), Fn<mul.<locals>.go>(i64))
(Fn<pipe(add.<locals>.go)>(Fn<add.<locals>.go>(i64), Fn<mul.<locals>.go>(i64)), Fn<pipe(add.<locals>.go)>(Fn<add.<locals>.go>(i64), Fn<mul.<locals>.go>(i64)))  <- pipelines compose as values, like any Handle


## What we can't do yet

- **Fusion is a promise, not a fact**: the `Derived("pipe", …)` build rule —
  emit the composed body as IR — lands with lowering (ch07), and `value >
  pipeline` computes for real at the CPU backend (ch09). From ch09 on,
  combinator style is the **house style** for examples.
- **Outputs**: `DeviceValue`, `ResultPlan`, and destination-passing arrive
  with bidirectional marshaling (ch08); materializers execute then.
- **Config** has its own specialization regime (040_combinators-notes §3c),
  deliberately NOT the capture thesis: per component, strip →
  value-specialize (default) → type-specialize (rare), schema-declared and
  kernel-overridable. Today, with no schemas, everything value-specializes —
  the general model's degenerate case.
- **Orchestration** (fragment-pass graphs, buffer chains) waits for the pass
  runtime, after the GPU chapter.

**Budget after step 3b:** kernel *unchanged* at 378 / 1150; the satellite is
133 counted lines outside the cap — that is the point.

**Next:** ch05 — programs are values: the `Node`/`Region` IR, content
hashing, and the invariant that makes the anti-pattern unrepresentable.